# Build Manifest

In [1]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_0_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

study folders: 0it [00:00, ?it/s]


KeyboardInterrupt



In [ ]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study-1/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_1_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

In [ ]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study-2/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_2_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

# Combine Dataframes

In [1]:
import pandas as pd
from pathlib import Path

LOCAL_DIR = Path("class_preds_es0")   # your local folder with preds_rank*.csv

paths = sorted(LOCAL_DIR.glob("preds_rank*.csv"))
print(f"found {len(paths)} files")

prob_cols = [
    "a2c","a3c","a4c","a5c","plax","tee","exclude",
    "psax-av","psax-mv","psax-ap","psax-pm"
]
dtypes = {"quality": "float32", "salience": "float32", **{f"p_{v}": "float32" for v in prob_cols}}

dfs = [pd.read_csv(p, dtype=dtypes) for p in paths]

es0 = pd.concat(dfs, ignore_index=True)
print(es0.shape)

found 8 files
(5843503, 16)


In [2]:
es0.head()

,png_uri,mp4_uri,pred_view,quality,salience,p_a2c,p_a3c,p_a4c,p_a5c,p_plax,p_tee,p_exclude,p_psax-av,p_psax-mv,p_psax-ap,p_psax-pm
0,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,exclude,0.042402,0.541041,0.000002,0.000120,0.000119,0.000952,0.222900,0.020416,0.754395,0.001119,0.000000,0.000137,0.000011
1,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,a2c,0.083963,0.576947,0.788086,0.080505,0.126953,0.000072,0.000019,0.002302,0.001903,0.000000,0.000000,0.000000,0.000000
2,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,tee,0.090764,0.362434,0.011147,0.000251,0.010651,0.000403,0.005856,0.478760,0.451660,0.000301,0.000902,0.012230,0.027878
3,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,a4c,0.062837,0.626273,0.094849,0.000040,0.867188,0.001674,0.000007,0.019531,0.016907,0.000000,0.000000,0.000000,0.000000
4,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,exclude,0.024978,0.703782,0.000076,0.000004,0.000198,0.003231,0.000531,0.001554,0.994629,0.000006,0.000000,0.000001,0.000000


In [3]:
import pandas as pd
from pathlib import Path

LOCAL_DIR = Path("class_preds_es1")   # your local folder with preds_rank*.csv

paths = sorted(LOCAL_DIR.glob("preds_rank*.csv"))
print(f"found {len(paths)} files")

prob_cols = [
    "a2c","a3c","a4c","a5c","plax","tee","exclude",
    "psax-av","psax-mv","psax-ap","psax-pm"
]
dtypes = {"quality": "float32", "salience": "float32", **{f"p_{v}": "float32" for v in prob_cols}}

dfs = [pd.read_csv(p, dtype=dtypes) for p in paths]

es1 = pd.concat(dfs, ignore_index=True)
print(es1.shape)

found 8 files
(770533, 16)


In [4]:
import pandas as pd
from pathlib import Path

LOCAL_DIR = Path("class_preds_es2")   # your local folder with preds_rank*.csv

paths = sorted(LOCAL_DIR.glob("preds_rank*.csv"))
print(f"found {len(paths)} files")

prob_cols = [
    "a2c","a3c","a4c","a5c","plax","tee","exclude",
    "psax-av","psax-mv","psax-ap","psax-pm"
]
dtypes = {"quality": "float32", "salience": "float32", **{f"p_{v}": "float32" for v in prob_cols}}

dfs = [pd.read_csv(p, dtype=dtypes) for p in paths]

es2 = pd.concat(dfs, ignore_index=True)
print(es2.shape)

found 8 files
(1228678, 16)


In [5]:
all_es = pd.concat([es0, es1, es2], ignore_index=True)

In [6]:
all_es.shape

(7842714, 16)

In [7]:
all_es.head()

,png_uri,mp4_uri,pred_view,quality,salience,p_a2c,p_a3c,p_a4c,p_a5c,p_plax,p_tee,p_exclude,p_psax-av,p_psax-mv,p_psax-ap,p_psax-pm
0,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,exclude,0.042402,0.541041,0.000002,0.000120,0.000119,0.000952,0.222900,0.020416,0.754395,0.001119,0.000000,0.000137,0.000011
1,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,a2c,0.083963,0.576947,0.788086,0.080505,0.126953,0.000072,0.000019,0.002302,0.001903,0.000000,0.000000,0.000000,0.000000
2,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,tee,0.090764,0.362434,0.011147,0.000251,0.010651,0.000403,0.005856,0.478760,0.451660,0.000301,0.000902,0.012230,0.027878
3,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,a4c,0.062837,0.626273,0.094849,0.000040,0.867188,0.001674,0.000007,0.019531,0.016907,0.000000,0.000000,0.000000,0.000000
4,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,exclude,0.024978,0.703782,0.000076,0.000004,0.000198,0.003231,0.000531,0.001554,0.994629,0.000006,0.000000,0.000001,0.000000


In [8]:
import pandas as pd

# Allow pandas to display the full content of columns without truncation
pd.set_option('display.max_colwidth', None)

# Select and display the first 5 rows of the desired columns
all_es[['png_uri', 'mp4_uri']].head()

,png_uri,mp4_uri
0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.811753780.1.1703111362.8215013.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.811753780.1.1703111362.8215013.mp4
1,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714578744.1.1703111407.7964352.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714578744.1.1703111407.7964352.mp4
2,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703111411.10353378.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714512485.1.1703111411.10353378.mp4
3,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4
4,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714578744.1.1703111393.7964326.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714578744.1.1703111393.7964326.mp4


# Find Remaining

In [11]:
import boto3
import s3fs
import gzip
import tempfile
import os
import pandas as pd
from tqdm.auto import tqdm

# --- Reusable Clients (define once) ---
s3_client = boto3.client("s3")
s3_fs = s3fs.S3FileSystem()

def create_filtered_manifest(
    bucket: str,
    source_manifest_key: str,
    df_to_exclude: pd.DataFrame,
    uri_column: str,
    new_manifest_key: str = None
):
    """
    Reads a gzipped manifest from S3, removes lines present in a DataFrame column,
    and uploads a new gzipped manifest with the remaining lines.

    Args:
        bucket (str): The S3 bucket name.
        source_manifest_key (str): The S3 key for the original manifest file.
        df_to_exclude (pd.DataFrame): DataFrame containing the URIs to remove.
        uri_column (str): The name of the column in df_to_exclude that holds the URIs.
        new_manifest_key (str, optional): The S3 key for the output manifest. 
                                           If None, it's auto-generated by adding '_rem' 
                                           to the original filename.
    """
    # --- 1. Generate the output key if not provided ---
    if new_manifest_key is None:
        dir_name = os.path.dirname(source_manifest_key)
        base_name = os.path.basename(source_manifest_key)
        # Safely split filename from its first extension-like part
        name_part, *ext_parts = base_name.split('.', 1)
        new_base_name = f"{name_part}_rem"
        if ext_parts:
            new_base_name += "." + ext_parts[0]
        new_manifest_key = os.path.join(dir_name, new_base_name)
    
    print(f"Source manifest: s3://{bucket}/{source_manifest_key}")
    print(f"Output manifest: s3://{bucket}/{new_manifest_key}")

    # --- 2. Create a set of URIs to remove for fast lookups ---
    print(f"\nCreating a set of URIs to exclude from column '{uri_column}'...")
    if uri_column not in df_to_exclude.columns:
        raise ValueError(f"The DataFrame must contain a '{uri_column}' column.")
    
    uris_to_remove = set(df_to_exclude[uri_column])
    print(f"Found {len(uris_to_remove):,} unique URIs to remove.")

    # --- 3. Stream, filter, and write to a new local manifest ---
    lines_read = 0
    lines_written = 0
    tmpf = tempfile.NamedTemporaryFile("wb", delete=False, suffix=".txt.gz")
    tmpf_path = tmpf.name
    print(f"\nProcessing and writing to temporary file: {tmpf_path}")

    try:
        with s3_fs.open(f's3://{bucket}/{source_manifest_key}', 'rb') as remote_file:
            with gzip.GzipFile(fileobj=remote_file, mode='rb') as gz_remote:
                with gzip.GzipFile(fileobj=tmpf, mode='wb', compresslevel=6) as gz_local:
                    pbar = tqdm(desc="Filtering manifest", unit="B", unit_scale=True, leave=False)
                    for line_bytes in gz_remote:
                        lines_read += 1
                        uri = line_bytes.decode('utf-8').strip()
                        if uri not in uris_to_remove:
                            lines_written += 1
                            gz_local.write(line_bytes) # Write original bytes directly
                        pbar.update(len(line_bytes))
                    pbar.close()
    finally:
        tmpf.close()

    print("\nFiltering complete.")
    print(f"  - Lines read from original: {lines_read:,}")
    print(f"  - Lines written to new:     {lines_written:,}")

    # --- 4. Upload the new manifest to S3 ---
    print(f"\nUploading new manifest to S3...")
    try:
        s3_client.upload_file(tmpf_path, bucket, new_manifest_key)
        print(f"✓ Upload successful.")
    except Exception as e:
        print(f"✗ Upload failed: {e}")
    finally:
        # --- 5. Clean up the local temporary file ---
        os.unlink(tmpf_path)
        print(f"✓ Temporary file removed.")

In [ ]:
# -------------------- EXAMPLE USAGE --------------------
# Assume 'es0' is your DataFrame loaded in the environment

# Name of the DataFrame column containing the URIs to be removed
uri_column_to_check = 'png_uri' 

if 'es0' in locals() and isinstance(es0, pd.DataFrame):
    create_filtered_manifest(
        bucket="echodata25",
        source_manifest_key="results/echo-images/all_unmasked_png_paths_0_v2.clean.txt.gz",
        df_to_exclude=es0,
        uri_column=uri_column_to_check
    )
else:
    print("Please define the DataFrame 'es0' to run the example.")

In [12]:
uri_column_to_check = 'png_uri' 

if 'es1' in locals() and isinstance(es0, pd.DataFrame):
    create_filtered_manifest(
        bucket="echodata25",
        source_manifest_key="results/echo-images/all_unmasked_png_paths_1_v2.clean.txt.gz",
        df_to_exclude=es1,
        uri_column=uri_column_to_check
    )
else:
    print("Please define the DataFrame 'es1' to run the example.")

Source manifest: s3://echodata25/results/echo-images/all_unmasked_png_paths_1_v2.clean.txt.gz
Output manifest: s3://echodata25/results/echo-images/all_unmasked_png_paths_1_v2_rem.clean.txt.gz

Creating a set of URIs to exclude from column 'png_uri'...
Found 770,533 unique URIs to remove.

Processing and writing to temporary file: /tmp/tmp3t6u6gfq.txt.gz


Filtering manifest: 0.00B [00:00, ?B/s]


Filtering complete.
  - Lines read from original: 1,584,665
  - Lines written to new:     814,132

Uploading new manifest to S3...
✓ Upload successful.
✓ Temporary file removed.


In [13]:
uri_column_to_check = 'png_uri' 

if 'es2' in locals() and isinstance(es0, pd.DataFrame):
    create_filtered_manifest(
        bucket="echodata25",
        source_manifest_key="results/echo-images/all_unmasked_png_paths_2_v2.clean.txt.gz",
        df_to_exclude=es2,
        uri_column=uri_column_to_check
    )
else:
    print("Please define the DataFrame 'es2' to run the example.")

Source manifest: s3://echodata25/results/echo-images/all_unmasked_png_paths_2_v2.clean.txt.gz
Output manifest: s3://echodata25/results/echo-images/all_unmasked_png_paths_2_v2_rem.clean.txt.gz

Creating a set of URIs to exclude from column 'png_uri'...
Found 1,228,678 unique URIs to remove.

Processing and writing to temporary file: /tmp/tmpyr30xsaf.txt.gz


Filtering manifest: 0.00B [00:00, ?B/s]


Filtering complete.
  - Lines read from original: 5,059,180
  - Lines written to new:     3,830,502

Uploading new manifest to S3...
✓ Upload successful.
✓ Temporary file removed.


# Fix MP4s

Check if the MP4s actually exist.

In [14]:
import pandas as pd
import boto3
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

S3 = boto3.client("s3")

def list_all_keys(bucket: str, prefix: str) -> list[str]:
    """List all object keys under one prefix."""
    paginator = S3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        keys.extend(obj["Key"] for obj in page.get("Contents", []))
    return keys

def _strip_bucket(uri: str, bucket: str) -> str:
    # s3://bucket/KEY... -> KEY...
    return uri.split(f"s3://{bucket}/", 1)[1]

def _check_prefix_group(args):
    """Worker: for one prefix, mark which rows exist."""
    prefix_uri, sub_df, bucket, col = args

    expected_keys = sub_df[col].map(lambda u: _strip_bucket(u, bucket))
    prefix_key   = _strip_bucket(prefix_uri, bucket)

    actual = set(list_all_keys(bucket, prefix_key))
    return expected_keys.isin(actual)

def mark_exists_by_listing(df: pd.DataFrame,
                           col: str = "mp4_uri",
                           workers: int = 64,
                           limit: int | None = None,
                           subset_only: bool = True) -> pd.Series:
    """
    Check S3 existence for the first `limit` rows (or all if None).
    Returns a boolean Series aligned to the checked subset (subset_only=True),
    or to the full df (subset_only=False, unchecked rows -> False).
    """
    dfc = df.iloc[:limit].copy() if limit is not None else df.copy()
    if dfc.empty:
        return pd.Series(dtype=bool)

    # assume single bucket
    bucket = dfc[col].iloc[0].split("/", 3)[2]

    # parent prefix of each file
    dfc["_prefix"] = dfc[col].map(lambda u: u.rsplit("/", 1)[0] + "/")

    tasks = [(p, sub, bucket, col) for p, sub in dfc.groupby("_prefix")]

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool, \
         tqdm(total=len(tasks), desc="prefixes", unit="pfx") as bar:
        futs = {pool.submit(_check_prefix_group, t): t for t in tasks}
        for fut in as_completed(futs):
            results.append(fut.result())
            bar.update(1)

    mask_subset = pd.concat(results).sort_index() if results else pd.Series(dtype=bool)

    if subset_only:
        return mask_subset

    # expand to full df
    mask_full = pd.Series(False, index=df.index)
    mask_full.loc[mask_subset.index] = mask_subset
    return mask_full

# -------- usage --------
mask = mark_exists_by_listing(all_es, limit=1000, subset_only=True)
missing = all_es.iloc[:len(mask)].loc[~mask, "mp4_uri"]
print(f"Checked {len(mask)} rows. Missing: {missing.size}")


prefixes:   0%|          | 0/132 [00:00<?, ?pfx/s]

Checked 1000 rows. Missing: 1000


# Fix MP4 Paths

Check which directories have more than one subdir.

In [9]:
# import boto3, time
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm.auto import tqdm

# # --- Configuration ---
# BUCKET            = "echodata25"
# ROOT_PREFIX       = "results/echo-study/"
# DCM_SERIES_PREFIX = "1.2.276"
# THREADS           = 64
# STUDY_PAGE        = 1_000
# SCAN_LIMIT        = None # Set to a number (e.g., 1000) to test on a subset

# s3 = boto3.client("s3")

# # ────────────────────────── 1. collect study prefixes ───────────────────────
# study_pref = []
# # Using a paginator is robust for listing all study folders
# for page in s3.get_paginator("list_objects_v2").paginate(
#         Bucket=BUCKET, Prefix=ROOT_PREFIX, Delimiter="/",
#         PaginationConfig={'PageSize': STUDY_PAGE}):
#     study_pref += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

# print(f"Found {len(study_pref):,} total studies.")

# if SCAN_LIMIT:
#     print(f"⚠️  Limiting scan to the first {SCAN_LIMIT:,} studies.")
#     study_pref = study_pref[:SCAN_LIMIT]

# # ──── 2. Find studies with >1 DICOM series directory in parallel ─────────
# def check_for_multiple_dicom_dirs(prefix):
#     """
#     Checks if a study has more than one subdirectory starting with '1.2.276'.
#     Returns the prefix if it matches, otherwise None.
#     """
#     dicom_dir_count = 0
#     paginator = s3.get_paginator("list_objects_v2")
#     try:
#         # List immediate subdirectories of the study
#         for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix, Delimiter="/"):
#             for cp in page.get("CommonPrefixes", []):
#                 # Extract subdirectory name, e.g., "1.2.276.xyz"
#                 sub_dir_name = cp["Prefix"].strip("/").split("/")[-1]
#                 if sub_dir_name.startswith(DCM_SERIES_PREFIX):
#                     dicom_dir_count += 1
                
#                 # Optimization: stop scanning as soon as the condition is met
#                 if dicom_dir_count > 1:
#                     raise StopIteration
#     except StopIteration:
#         return prefix # Condition met, return the study prefix
    
#     return None # Condition not met after checking all subdirectories

# matching_studies = []
# t0 = time.time()
# with ThreadPoolExecutor(max_workers=THREADS) as pool:
#     futures = {pool.submit(check_for_multiple_dicom_dirs, p) for p in study_pref}
#     for fut in tqdm(as_completed(futures), total=len(futures), desc="Scanning studies"):
#         result = fut.result()
#         if result:
#             matching_studies.append(result)

# elapsed = time.time() - t0

# # ────────────────────────── 3. Print the results ────────────────────────────
# print("\n" + "="*80)
# print(f"Scan complete in {elapsed:.1f} seconds.")
# if matching_studies:
#     print(f"✅ Found {len(matching_studies)} studies with more than one '{DCM_SERIES_PREFIX}...' subdirectory:")
#     for prefix in sorted(matching_studies):
#         print(f"s3://{BUCKET}/{prefix}")
# else:
#     print(f"ℹ️ No studies found matching the criteria.")
# print("="*80)

Fix the MP4 paths.

In [16]:
import re
import os
import pandas as pd
import boto3
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# -------------------- config --------------------
URI_COL   = "png_uri"
OUT_COL   = "mp4_uri_corrected"
WORKERS   = 64
BUCKET_RE = re.compile(r"^s3://([^/]+)/")
S3 = boto3.client("s3")

# -------------------- helpers (unchanged) --------------------
def s3_bucket_key(uri: str) -> tuple[str, str]:
    m = BUCKET_RE.match(uri)
    if not m:
        raise ValueError(f"bad s3 uri: {uri}")
    b = m.group(1)
    return b, uri[len(f"s3://{b}/"):]

def study_root_from_png(uri: str) -> str:
    return uri.split("/unmasked/", 1)[0] + "/"

def basename_no_ext(uri: str) -> str:
    return os.path.basename(uri).rsplit(".", 1)[0]

def list_all_keys(bucket: str, prefix: str) -> list[str]:
    paginator = S3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if "Contents" in page:
             keys.extend(obj["Key"] for obj in page.get("Contents", []))
    return keys

def build_mapping_for_study(study_prefix: str, bucket: str) -> dict[str, str]:
    prefix_key = study_prefix.replace(f"s3://{bucket}/", "", 1)
    all_keys = list_all_keys(bucket, prefix_key)
    mp4_keys = [
        k for k in all_keys
        if k.endswith(".mp4")
        and "/unmasked/" not in k
        and not k.endswith("mask_visualization.mp4")
    ]
    return {os.path.basename(k)[:-4]: f"s3://{bucket}/{k}" for k in mp4_keys}

# -------------------- main (modified) --------------------
def map_png_to_mp4(df: pd.DataFrame,
                   uri_col: str = URI_COL,
                   out_col: str = OUT_COL,
                   workers: int = WORKERS,
                   limit: int = None,
                   view: str = None) -> pd.Series:
    if df.empty:
        return pd.Series(dtype=object, index=df.index)

    # --- Create a subset of the DataFrame to process based on filters ---
    df_to_process = df.copy()
    if view:
        if "pred_view" in df_to_process.columns:
            df_to_process = df_to_process[df_to_process["pred_view"] == view]
        else:
            print(f"Warning: 'pred_view' column not found. Cannot filter by view '{view}'.")
    if limit:
        df_to_process = df_to_process.head(limit)

    if df_to_process.empty:
        print("No rows match the specified filters.")
        return pd.Series(dtype=object, index=df.index)
    
    print(f"Processing {len(df_to_process):,} filtered rows...")

    # The rest of the function operates on the filtered `df_to_process`
    bucket = BUCKET_RE.match(df_to_process[uri_col].iloc[0]).group(1)

    tmp = pd.DataFrame({
        "uri": df_to_process[uri_col].values,
        "study_prefix": df_to_process[uri_col].map(study_root_from_png),
        "basename": df_to_process[uri_col].map(basename_no_ext),
    }, index=df_to_process.index)

    groups = list(tmp.groupby("study_prefix"))
    
    mappings = {}
    with ThreadPoolExecutor(max_workers=workers) as pool, \
         tqdm(total=len(groups), desc="list S3", unit="study") as pbar:
        futs = {
            pool.submit(build_mapping_for_study, study, bucket): study
            for study, _ in groups
        }
        for fut in as_completed(futs):
            study = futs[fut]
            mappings[study] = fut.result()
            pbar.update(1)

    def resolve(row):
        return mappings.get(row["study_prefix"], {}).get(row["basename"], None)

    # Apply the mapping to the filtered subset
    processed_results = tmp.apply(resolve, axis=1)

    # Reindex to match the original DataFrame, filling non-processed rows with NaN
    return processed_results.reindex(df.index)

# -------------------- usage --------------------
# Example: Correct only the first 1000 rows that are predicted as 'a4c'
a4c_limit = 10000
all_es["mp4_uri_corrected_a4c"] = map_png_to_mp4(
    all_es,
    limit=a4c_limit,
    view='a4c'
)

# Example: Correct all 'a4c' rows (no limit)
# all_es["mp4_uri_corrected_a4c_all"] = map_png_to_mp4(all_es, view='a4c')

# Example: Correct all rows (original behavior)
# all_es["mp4_uri_corrected_all"] = map_png_to_mp4(all_es)

mapped_count = all_es['mp4_uri_corrected_a4c'].notna().sum()
print(f"Mapped {mapped_count:,} URIs for view 'a4c' with a limit of {a4c_limit}.")

Processing 10,000 filtered rows...


list S3:   0%|          | 0/5700 [00:00<?, ?study/s]

Mapped 10,000 URIs for view 'a4c' with a limit of 10000.


In [17]:
# Assuming 'mp4_uri_corrected_a4c' is the new column you created
corrected_rows_only_df = all_es[all_es['mp4_uri_corrected_a4c'].notna()].copy()

# You can now view the new DataFrame
print(f"Found {len(corrected_rows_only_df)} successfully corrected rows.")
corrected_rows_only_df.head()

Found 10000 successfully corrected rows.


,png_uri,mp4_uri,pred_view,quality,salience,p_a2c,p_a3c,p_a4c,p_a5c,p_plax,p_tee,p_exclude,p_psax-av,p_psax-mv,p_psax-ap,p_psax-pm,mp4_uri_corrected_a4c
3,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4,a4c,0.062837,0.626273,0.094849,0.000040,0.867188,0.001674,0.000007,0.019531,0.016907,0.000000,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4
5,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.mp4,a4c,0.041382,0.408655,0.022949,0.000096,0.565918,0.000573,0.000153,0.000055,0.410156,0.000004,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/1.2.276.0.7230010.3.1.3.1714512485.1.1703119859.10459774/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.mp4
9,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.811753780.1.1703111413.8215093.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.811753780.1.1703111413.8215093.mp4,a4c,0.059262,0.619341,0.043915,0.007721,0.858887,0.000662,0.000402,0.064148,0.024017,0.000000,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111413.8215093.mp4
11,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.811753780.1.1703111387.8215053.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.811753780.1.1703111387.8215053.mp4,a4c,0.056213,0.623309,0.048645,0.000160,0.866211,0.001134,0.002815,0.020248,0.060638,0.000001,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111387.8215053.mp4
13,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.mp4,a4c,0.125076,0.735765,0.001447,0.000007,0.997070,0.000363,0.000000,0.000001,0.001352,0.000000,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.mp4


In [18]:
# 1. Extract the DeidentifiedStudyID from the 'png_uri' column
# This splits the string and takes the part after '/echo-study/' and before the next '/'
corrected_rows_only_df['DeidentifiedStudyID'] = corrected_rows_only_df['png_uri'].str.split('/echo-study/').str[1].str.split('/').str[0]

# 2. Create the final DataFrame with only the desired columns
# and rename 'mp4_uri_corrected_a4c' to 'URI'
final_df = corrected_rows_only_df[['DeidentifiedStudyID', 'mp4_uri_corrected_a4c']].rename(
    columns={'mp4_uri_corrected_a4c': 'URI'}
)

In [20]:
final_df.head()

,DeidentifiedStudyID,URI
3,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4
5,1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/1.2.276.0.7230010.3.1.3.1714512485.1.1703119859.10459774/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.mp4
9,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111413.8215093.mp4
11,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111387.8215053.mp4
13,1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.mp4


In [21]:
# 3. Save the result to a CSV file
output_filename = 'a4c_10k.csv'
final_df.to_csv(output_filename, index=False)

print(f"✅ Successfully saved {len(final_df)} rows to '{output_filename}'")
print("\nPreview of the final data:")
final_df.head()

✅ Successfully saved 10000 rows to 'a4c_10k.csv'

Preview of the final data:


,DeidentifiedStudyID,URI
3,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4
5,1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/1.2.276.0.7230010.3.1.3.1714512485.1.1703119859.10459774/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.mp4
9,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111413.8215093.mp4
11,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111387.8215053.mp4
13,1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.mp4


# MVR

In [22]:
mvr_labels = pd.read_csv('mvr_labels.csv')

In [23]:
mvr_labels.head()

,Unnamed: 0,STUDY_REF,s3_key,Value,STUDY_DATE,STUDY_TIME,DeidentifiedStudyID,OriginalStudyID
0,0,1000750,echo-study-2/1.2.276.0.7230010.3.1.2.895693665.1.1725195593.4282089/,mild,20150722,80635.0,1.2.276.0.7230010.3.1.2.895693665.1.1725195593.4282089,1.3.12.2.1107.5.8.9.1002655211149138.20150722120043279
1,1,1000846,echo-study-2/1.2.276.0.7230010.3.1.2.842097970.1.1725195860.2750974/,trace,20150722,93052.0,1.2.276.0.7230010.3.1.2.842097970.1.1725195860.2750974,1.3.12.2.1107.5.8.9.1002655211149138.20150722131319960
2,2,1000847,echo-study-2/1.2.276.0.7230010.3.1.2.842097970.1.1725195931.2751236/,trace,20150722,93351.0,1.2.276.0.7230010.3.1.2.842097970.1.1725195931.2751236,1.3.12.2.1107.5.8.9.1002655211149138.20150722132025828
3,3,1000712,echo-study-2/1.2.276.0.7230010.3.1.2.1714500150.1.1725195543.2219892/,moderate,20150722,80437.0,1.2.276.0.7230010.3.1.2.1714500150.1.1725195543.2219892,1.3.12.2.1107.5.8.9.1002655211149138.20150722112601485
4,4,1000714,echo-study-2/1.2.276.0.7230010.3.1.2.895693665.1.1725195567.4281989/,trace,20150722,75829.0,1.2.276.0.7230010.3.1.2.895693665.1.1725195567.4281989,1.3.12.2.1107.5.8.9.1002655211149138.20150722115621748


In [24]:
len(mvr_labels)

45152

In [25]:
# Select only the necessary columns from your labels DataFrame for efficiency
labels_to_merge = mvr_labels[['DeidentifiedStudyID', 'Value']]

# Merge the two DataFrames to find the overlap and add the 'Value'
# 'how="inner"' ensures only matching DeidentifiedStudyIDs are kept
merged_df = pd.merge(
    final_df, 
    labels_to_merge, 
    on='DeidentifiedStudyID', 
    how='inner'
)

print(f"Found {len(merged_df)} overlapping studies.")
merged_df.head()

Found 0 overlapping studies.


,DeidentifiedStudyID,URI,Value
